## Word2Vec

In [ ]:
import os

import torch
from models import CBOW, SkipGram

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    device = torch.device(device)
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )
    vocab_size = checkpoint["vocab_size"]
    embedding_dim = checkpoint["embedding_dim"]
    internal_node_count = checkpoint["internal_node_count"]

    if checkpoint["mode"] == "cbow":
        model = CBOW(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            internal_node_count=internal_node_count,
            padding_idx=checkpoint.get("padding_idx", vocab_size - 1),
        )
    elif checkpoint["mode"] == "skip-gram":
        model = SkipGram(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            internal_node_count=internal_node_count,
        )

    model.load_state_dict(checkpoint["word2vec"])
    model = model.to(device)
    model.eval()

    word2idx = checkpoint["word2idx"]
    idx2word = checkpoint["idx2word"]
    pad_idx = checkpoint.get("padding_idx", checkpoint.get("pad_idx"))
    return model, word2idx, idx2word, pad_idx, vocab_size

### WikiText-2

#### Continuous Bag of Words

In [ ]:
wikitext2_cbow_path = os.path.join("checkpoints", "wikitext2", "wikitext2-cbow_20.pth")

In [ ]:
wikitext2_cbow_model, wikitext2_cbow_word2idx, wikitext2_cbow_idx2word, wikitext2_cbow_pad_idx, wikitext2_cbow_vocab_size = load_checkpoint(wikitext2_cbow_path, device=device)

In [ ]:
word = input()
print(word)

In [ ]:
word2idx = wikitext2_cbow_word2idx
idx2word = wikitext2_cbow_idx2word

top_k = 3

embeddings = torch.nn.functional.normalize(
    wikitext2_cbow_model.embed.weight.detach(), dim=1
)
scores = embeddings @ embeddings[word2idx[word]]
scores[word2idx[word]] = -torch.inf
if wikitext2_cbow_pad_idx is not None:
    scores[wikitext2_cbow_pad_idx] = -torch.inf

neighbors = [
	idx2word[index]
	for index in scores.topk(top_k).indices.tolist()
]
print(neighbors)

#### Skip-Gram

In [ ]:
wikitext2_skipgram_path = os.path.join("checkpoints", "wikitext2", "wikitext2-skip-gram_20.pth")

In [ ]:
wikitext2_skipgram_model, wikitext2_skipgram_word2idx, wikitext2_skipgram_idx2word, wikitext2_skipgram_pad_idx, wikitext2_skipgram_vocab_size = load_checkpoint(wikitext2_skipgram_path, device=device)

In [ ]:
word = input()
print(word)

In [ ]:
word2idx = wikitext2_skipgram_word2idx
idx2word = wikitext2_skipgram_idx2word

top_k = 3

embeddings = torch.nn.functional.normalize(
    wikitext2_skipgram_model.embed.weight.detach(), dim=1
)
scores = embeddings @ embeddings[word2idx[word]]
scores[word2idx[word]] = -torch.inf
if wikitext2_skipgram_pad_idx is not None:
    scores[wikitext2_skipgram_pad_idx] = -torch.inf

neighbors = [
	idx2word[index]
	for index in scores.topk(top_k).indices.tolist()
]
print(neighbors)